#### READ THE PRODUCT CATEGORY DATA FROM VOLUME

In [0]:
BASE_DIR = '/Volumes/orders_sales/landing/operational/Product_Category_Translation/'
CHECKPOINT_DIR = '/Volumes/orders_sales/landing/checkpoint_dir'
CHECKPOINT_LOCATION = f'{CHECKPOINT_DIR}/product_category_translation'
CATALOG = 'ORDERS_SALES'
SCHEMA = 'BRONZE'
TARGET_TABLE = 'PRODUCT_CATEGORY'
print(f'Checkpoint Location: {CHECKPOINT_LOCATION}')
print(f'Base Location: {BASE_DIR}')

In [0]:

product_category_schema = """product_category_name STRING, 
                 product_category_name_english STRING
                 """

#### LOAD THE DATA INTO BRONZE LAYER
- `ORDERS_SALES.BRONZE.PRODUCT_CATEGORY`

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

product_category_df  = (
            spark.readStream.format('cloudFiles')
                 .option('cloudFiles.format', 'csv')
                 .option('cloudFiles.schemaEvolutionMode', 'rescue')
                 .option('header', 'true')
                 .schema(product_category_schema)
                 .load(f'{BASE_DIR}')
        )
product_category_trans_df = product_category_df.select(
    "product_category_name",
    "product_category_name_english",
    "_metadata.file_name",
    "_metadata.file_path",).withColumn('load_timestamp', current_timestamp())

# product_category_trans_df.printSchema()
# product_category_trans_df.limit(1).display()

qry = (
    product_category_trans_df.writeStream.format('delta')
    .outputMode('append')
    .option('checkpointLocation', CHECKPOINT_LOCATION)
    .trigger(availableNow=True)
    .toTable(f'{CATALOG}.{SCHEMA}.{TARGET_TABLE}')
)
qry.awaitTermination()